# Static Hand Gesture Recognition — run every experiment (Colab / Kaggle)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/hand-gesture-recognition/blob/main/notebooks/run_all_experiments.ipynb)

This notebook trains and evaluates **all models** (MLP, HOG-MLP, custom CNN, MobileNetV2, ResNet18 + ablations) on LeapGestRecog under one protocol, then builds the comparison table and plots.

**Before running:** `Runtime → Change runtime type → GPU (T4)`. Total time on a T4 ≈ 30–45 min.

On **Kaggle**: create a notebook, *Add Input → Datasets → gti-upm/leapgestrecog*, enable GPU + Internet, and run the cells below.

## 1 · Get the code

In [ ]:
REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/hand-gesture-recognition.git"  # <- change me

import os, sys
if not os.path.exists("src/train.py"):
    if not os.path.exists("hand-gesture-recognition"):
        !git clone -q $REPO_URL hand-gesture-recognition
    %cd hand-gesture-recognition
!pip -q install -r requirements.txt
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 2 · Get the dataset
On Kaggle with the dataset attached this just links it. On Colab, `kagglehub` needs your Kaggle API token: either add `KAGGLE_USERNAME` / `KAGGLE_KEY` as Colab secrets (🔑 icon in the left bar), or upload `kaggle.json` when prompted.

In [ ]:
import os
if not os.path.exists("/kaggle/input") and not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
    try:
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    except Exception:
        try:
            from google.colab import files
            print("Upload kaggle.json (Kaggle → Settings → API → Create New Token)")
            up = files.upload()
            os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
            open(os.path.expanduser("~/.kaggle/kaggle.json"), "wb").write(up["kaggle.json"])
            os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
        except Exception as e:
            print("Set up Kaggle credentials manually:", e)
!python scripts/download_data.py

## 3 · Sanity check (pytest smoke tests, ~30 s)

In [ ]:
!python -m pytest -q tests

## 4 · Train + evaluate every model
Each run saves `checkpoints/<run>.pt` and `results/<run>/` (metrics JSON, confusion matrix, curves). `--skip-existing` lets you re-run this cell to resume if the session disconnects.

In [ ]:
!python scripts/run_all.py --skip-existing --num-workers 2

## 5 · Comparison

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open("results/comparison.md").read()))
for f in ["comparison_accuracy_f1.png", "comparison_accuracy_vs_latency.png", "comparison_per_class_f1.png"]:
    display(Image(f"results/{f}"))

In [ ]:
# Confusion matrices and training curves of every run
import glob
for d in sorted(glob.glob("results/*/")):
    print(d)
    display(Image(d + "confusion_matrix_test.png", width=420), Image(d + "training_curves.png", width=620))

## 6 · (Optional) random-split comparison
Shows how much a naive random frame split inflates accuracy, because near-identical consecutive frames of the same person leak into the test set.

In [ ]:
# !python -m src.train --model cnn --split random --run-name cnn_randomsplit
# !python -m src.compare

## 7 · Download everything to push to GitHub
Downloads `results/`, `checkpoints/` and the updated `README.md` (with the filled-in results table) as one zip. Unzip it into your local repo folder (overwrite), then `git add . && git commit -m "Add results" && git push`.

In [ ]:
!zip -qr experiment_outputs.zip results checkpoints README.md
try:
    from google.colab import files; files.download("experiment_outputs.zip")
except Exception:
    print("Saved experiment_outputs.zip (on Kaggle: find it under Output)")